# Data Collection — Wikipedia Web Scraping
**Purpose:** collect historical Falcon 9 launch table data from Wikipedia using `requests` and BeautifulSoup.

The page structure may change. The notebook prints the tables it finds so the extraction can be checked rather than silently assuming a fixed table index.

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from IPython.display import display

URL = "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches"
headers = {"User-Agent": "Mozilla/5.0 (compatible; educational data science project)"}
response = requests.get(URL, headers=headers, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")
tables = soup.find_all("table", class_="wikitable")
print("Wikipedia tables found:", len(tables))

In [ ]:
frames = []
for idx, table in enumerate(tables):
    try:
        frame = pd.read_html(str(table))[0]
        if not frame.empty:
            frame["source_table_index"] = idx
            frames.append(frame)
    except ValueError:
        continue

print("Tables successfully parsed:", len(frames))
if not frames:
    raise RuntimeError("No HTML tables were parsed. Inspect the page structure or try again later.")
display(frames[0].head())

In [ ]:
# Combine parsed tables with a source index; columns can differ across historical tables.
scraped_df = pd.concat(frames, ignore_index=True, sort=False)
print("Combined shape:", scraped_df.shape)
display(scraped_df.head())
scraped_df.to_csv("wikipedia_falcon9_launch_tables.csv", index=False)
print("Saved: wikipedia_falcon9_launch_tables.csv")

## Interpretation
Web scraping converts HTML tables into a tabular dataset for further inspection. Before using this file for modelling, standardize column names, inspect duplicate/header rows, and verify that outcome labels are consistent. The scraped tables are supporting historical data; do not assume they are directly interchangeable with the course's modelling dataset.